# Импорты и базовые настройки

In [1]:
import os
import re
import sys
import random
import subprocess
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def safe_ensure_package(package_name: str, import_name: Optional[str] = None) -> bool:
    """Пытается импортировать пакет и при необходимости установить его через pip.
    Если установка не удалась, возвращает False, но не роняет ноутбук.
    """
    target = import_name or package_name
    try:
        __import__(target)
        return True
    except Exception:
        print(f"Пробуем установить пакет: {package_name}")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
            __import__(target)
            return True
        except Exception as e:
            print(f"Не удалось подготовить пакет {package_name}: {e!r}")
            return False


FAISS_READY = safe_ensure_package("faiss-cpu", "faiss")

try:
    import faiss  # type: ignore
    FAISS_AVAILABLE = True
except Exception:
    faiss = None
    FAISS_READY = False
    FAISS_AVAILABLE = False

CHUNK_SIZE = 40
OVERLAP = 12


# sentence-transformers опционален: ноутбук умеет работать и без него.
SENTENCE_TRANSFORMERS_READY = safe_ensure_package("sentence-transformers", "sentence_transformers")

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS доступен:", FAISS_READY)
print("sentence-transformers доступен:", SENTENCE_TRANSFORMERS_READY)

NumPy: 2.4.3
Pandas: 3.0.1
FAISS доступен: True
sentence-transformers доступен: True


In [2]:
# Фиксируем seed и определяем устройство.
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)


set_seed(42)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("Устройство для работы:", DEVICE)

Устройство для работы: cuda


# База знаний и первичный анализ

In [3]:
text_labels = ["GNU Automake - Introduction",
               "An introduction to the Autotools",
               "Basic installation with GNU Autotools",
               "How Autotools aid development",
               "General operation of GNU Autotools",
               "Standard Makefile Targets in Autotools",
               "Standard Directory Variables in Autotools",
               "Standard Configuration Variables in Autotools",
               "GNU Autotools Parallel Build Trees",
               "Cross Compilation with GNU Autotools",
              ]
text_labels_added = [
    "GNU Autotools Automatic Dependency Tracking",
    "GNU Autoconf",
]
texts = [open(f"data/doc-{i}.txt","r").read() for i in range(10)]
for i in range(len(texts)):
    print(f"{text_labels[i]}: {texts[i]}\n")
texts_added = [open(f"data/doc-{i}.txt","r").read() for i in range(10,12)]

GNU Automake - Introduction: Automake is a tool for automatically generating Makefile.ins from files called Makefile.am. Each Makefile.am is basically a series of make variable definitions1, with rules being thrown in occasionally. The generated Makefile.ins are compliant with the GNU Makefile standards. The GNU Makefile Standards Document is long, complicated, and subject to change. The goal of Automake is to remove the burden of Makefile maintenance from the back of the individual GNU maintainer (and put it on the back of the Automake maintainers). The typical Automake input file is simply a series of variable definitions. Each such file is processed to create a Makefile.in. Automake does constrain a project in certain ways; for instance, it assumes that the project uses Autoconf (see Introduction in The Autoconf Manual), and enforces certain restrictions on the configure.ac contents. Automake requires perl in order to generate the Makefile.ins. However, the distributions created by 

In [4]:
documents: List[Dict[str, str]] = [ {"doc_id":f"doc_{i}", "title":text_labels[i], "text": texts[i]} for i in range(len(text_labels))]

In [5]:
docs_df = pd.DataFrame(documents)
display(docs_df[["doc_id", "title","text"]])

,doc_id,title,text
0,doc_0,GNU Automake - Introduction,Automake is a tool for automatically generatin...
1,doc_1,An introduction to the Autotools,"It is a truth universally acknowledged, that a..."
2,doc_2,Basic installation with GNU Autotools,"The user first unpacks the package. Here, and ..."
3,doc_3,How Autotools aid development,There are several reasons why you may not want...
4,doc_4,General operation of GNU Autotools,Automake works by reading a Makefile.am and ge...
5,doc_5,Standard Makefile Targets in Autotools,So far we have come across four ways to run ma...
6,doc_6,Standard Directory Variables in Autotools,The GNU Coding Standards also specify a hierar...
7,doc_7,Standard Configuration Variables in Autotools,The GNU Coding Standards also define a set of ...
8,doc_8,GNU Autotools Parallel Build Trees,The GNU Build System distinguishes two trees: ...
9,doc_9,Cross Compilation with GNU Autotools,To cross-compile is to build on one platform a...


#### Документация GNU Autotools - хаотичное побоище, в котором редко можно понять, что происходит. Предметная область - аспекты использования утилит автоматизации сборки GNU Build Tools (или GNU Autotools). Здесь уместно использовать mini-Rag/retrieval, возможно, чтобы упорядочить материал по использованию Autotools или чтобы получить быстрый ответ на вопрос.

# Чанкинг документов

In [6]:
def chunk_text(text: str, chunk_size: int = 30, overlap: int = 10) -> List[str]:
    words = text.split()
    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk_words = words[start:end]
        if not chunk_words:
            continue
        chunks.append(" ".join(chunk_words))
        if end >= len(words):
            break
    return chunks


class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError


class TfidfBackend(EmbeddingBackend):
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        self.backend_name = "TF-IDF (fallback)"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        matrix = self.vectorizer.fit_transform(texts)
        vectors = matrix.astype(np.float32).toarray()
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        matrix = self.vectorizer.transform(texts)
        vectors = matrix.astype(np.float32).toarray()
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms


class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer  # type: ignore
        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        return vectors.astype(np.float32)

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        return vectors.astype(np.float32)


def choose_backend(device: str = "cpu") -> EmbeddingBackend:
    # Опциональная попытка dense backend.
    if SENTENCE_TRANSFORMERS_READY:
        try:
            return SentenceTransformersBackend(
                model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
                device=device,
            )
        except Exception as e:
            print("Dense backend недоступен, переходим к TF-IDF.")
            print("Причина:", repr(e))
    return TfidfBackend()


@dataclass
class RetrieverArtifacts:
    backend_name: str
    chunks_df: pd.DataFrame
    chunk_vectors: np.ndarray
    backend: EmbeddingBackend
    index: object


def build_retriever(
    documents: List[Dict[str, str]],
    chunk_size: int = 28,
    overlap: int = 8,
    device: str = "cpu",
) -> RetrieverArtifacts:
    rows = []
    for doc in documents:
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for chunk_id, chunk_text_value in enumerate(chunks, start=1):
            rows.append(
                {
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "chunk_id": chunk_id,
                    "chunk_text": chunk_text_value,
                    "n_words": len(chunk_text_value.split()),
                }
            )

    chunks_df = pd.DataFrame(rows)
    backend = choose_backend(device=device)
    chunk_vectors = backend.fit_documents(chunks_df["chunk_text"].tolist()).astype(np.float32)

    if FAISS_READY:
        index = faiss.IndexFlatIP(chunk_vectors.shape[1])  # type: ignore
        index.add(chunk_vectors)
    else:
        index = chunk_vectors

    return RetrieverArtifacts(
        backend_name=backend.backend_name,
        chunks_df=chunks_df,
        chunk_vectors=chunk_vectors,
        backend=backend,
        index=index,
    )


def search_chunks(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> pd.DataFrame:
    query_vector = artifacts.backend.encode_queries([query]).astype(np.float32)

    if FAISS_READY:
        scores, indices = artifacts.index.search(query_vector, top_k)  # type: ignore
        scores = scores[0]
        indices = indices[0]
    else:
        similarities = (artifacts.chunk_vectors @ query_vector.T).reshape(-1)
        indices = np.argsort(-similarities)[:top_k]
        scores = similarities[indices]

    result = artifacts.chunks_df.iloc[indices].copy().reset_index(drop=True)
    result.insert(0, "rank", np.arange(1, len(result) + 1))
    result["score"] = scores
    return result[["rank", "score", "doc_id", "title", "chunk_id", "chunk_text"]]

In [7]:
artifacts = build_retriever(
    documents,
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP,
    device=DEVICE,
)

print("Используемый backend:", artifacts.backend_name)
print("Количество чанков:", len(artifacts.chunks_df))
display(artifacts.chunks_df.head())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Используемый backend: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Количество чанков: 91


,doc_id,title,chunk_id,chunk_text,n_words
0,doc_0,GNU Automake - Introduction,1,Automake is a tool for automatically generatin...,40
1,doc_0,GNU Automake - Introduction,2,The generated Makefile.ins are compliant with ...,40
2,doc_0,GNU Automake - Introduction,3,remove the burden of Makefile maintenance from...,40
3,doc_0,GNU Automake - Introduction,4,file is simply a series of variable definition...,40
4,doc_0,GNU Automake - Introduction,5,assumes that the project uses Autoconf (see In...,40


In [8]:
sample_queries = [
    "What are the default Makefile targets for GNU Autotools?",
    "What does Automake do?",
    "Why would I not like to implement GNU Autotools?",
]

for query in sample_queries:
    display(Markdown(f"### Запрос: {query}"))
    display(search_chunks(query, artifacts=artifacts, top_k=3))

### Запрос: What are the default Makefile targets for GNU Autotools?

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.772912,doc_0,GNU Automake - Introduction,2,The generated Makefile.ins are compliant with ...
1,2,0.739728,doc_1,An introduction to the Autotools,14,GNU Build System for your package. Autoconf mo...
2,3,0.721650,doc_0,GNU Automake - Introduction,1,Automake is a tool for automatically generatin...


### Запрос: What does Automake do?

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.667475,doc_0,GNU Automake - Introduction,1,Automake is a tool for automatically generatin...
1,2,0.616378,doc_1,An introduction to the Autotools,13,GNU project. However it is used by a vast numb...
2,3,0.616334,doc_4,General operation of GNU Autotools,1,Automake works by reading a Makefile.am and ge...


### Запрос: Why would I not like to implement GNU Autotools?

,rank,score,doc_id,title,chunk_id,chunk_text
0,1,0.770509,doc_3,How Autotools aid development,7,there also exist reasons why you may want NOT ...
1,2,0.743484,doc_1,An introduction to the Autotools,13,GNU project. However it is used by a vast numb...
2,3,0.716883,doc_3,How Autotools aid development,5,Autotools take all this burden off your back a...


# Эмбеддинги и индекс FAISS

In [ ]:
embedder = choose_backend(device=DEVICE)

In [ ]:
# Строим векторные представления для всех чанков в ручную.
chunk_texts = artifacts.chunks_df["chunk_text"].tolist()
chunk_embeddings = embedder.fit_documents(chunk_texts)

print("Форма матрицы эмбеддингов:", chunk_embeddings.shape)

# Проверяем длины векторов.
# Если normalize_embeddings=True сработал корректно, все нормы должны быть ≈ 1.0.
# Это означает, что косинусное сходство далее можно считать через скалярное произведение.
vector_norms = np.linalg.norm(chunk_embeddings, axis=1)
print("Минимальная норма:", round(float(vector_norms.min()), 4))
print("Максимальная норма:", round(float(vector_norms.max()), 4))
print("Средняя норма:", round(float(vector_norms.mean()), 4))
print("→ Нормы ≈ 1.0: нормировка подтверждена, dot product = cosine similarity.")

In [ ]:
 # Считаем сходство запроса и всех чанков без FAISS.
query = "Why would I not like to implement GNU Autotools?"
query_vector = embedder.encode_queries([query])

# Для нормализованных векторов это эквивалент косинусного сходства.
manual_scores = (chunk_embeddings @ query_vector.T).ravel()

top_k = 5
top_indices = np.argsort(manual_scores)[::-1][:top_k]

manual_results_df = artifacts.chunks_df.iloc[top_indices].copy()
manual_results_df["score"] = manual_scores[top_indices]
manual_results_df["score"] = manual_results_df["score"].round(4)
manual_results_df.insert(0, "rank", range(1, len(manual_results_df) + 1))

display(Markdown(f"**Запрос:** {query}"))
display(manual_results_df[["rank", "doc_id", "title", "chunk_id", "score", "chunk_text"]])

In [ ]:
# Единая обёртка над FAISS и fallback-поиском.
class VectorSearchIndex:
    def __init__(self, dim: int) -> None:
        self.dim = dim
        self.backend_name = None
        self._faiss_index = None
        self._nn_index = None

        if FAISS_AVAILABLE:
            self._faiss_index = faiss.IndexFlatIP(dim)  # type: ignore[name-defined]
            self.backend_name = "FAISS IndexFlatIP"
        else:
            self._nn_index = NearestNeighbors(metric="cosine")
            self.backend_name = "sklearn NearestNeighbors fallback"

    def add(self, vectors: np.ndarray) -> None:
        vectors = vectors.astype("float32")

        if self._faiss_index is not None:
            self._faiss_index.add(vectors)
        else:
            self._nn_index.fit(vectors)

    def search(self, query_vectors: np.ndarray, top_k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        query_vectors = query_vectors.astype("float32")

        if self._faiss_index is not None:
            scores, indices = self._faiss_index.search(query_vectors, top_k)
            return scores, indices

        distances, indices = self._nn_index.kneighbors(query_vectors, n_neighbors=top_k)
        scores = 1.0 - distances
        return scores, indices


search_index = VectorSearchIndex(dim=chunk_embeddings.shape[1])
search_index.add(chunk_embeddings)

print("Индекс построен.")
print("Бэкэнд индекса:", search_index.backend_name)

In [ ]:
# Проверяем FAISS-поиск на первом запросе.
# Результаты должны совпасть с ручным расчётом из секции 6:
# тот же top-k, те же фрагменты, те же оценки сходства.
faiss_query = "How do i set linker flags?"
faiss_results_df = search_chunks(faiss_query, artifacts, top_k=5)

display(Markdown(f"**Запрос:** {faiss_query}"))
display(faiss_results_df)

In [ ]:
def unique_doc_order(result_df: pd.DataFrame) -> List[str]:
    seen = set()
    ordered = []
    for doc_id in result_df["doc_id"].tolist():
        if doc_id not in seen:
            seen.add(doc_id)
            ordered.append(doc_id)
    return ordered
def evaluate_query(
    query: str,
    relevant_doc_ids: List[str],
    artifacts: RetrieverArtifacts,
    top_k: int = 3,
) -> Dict[str, object]:
    result_df = search_chunks(query, artifacts, top_k=top_k)
    predicted_doc_ids = unique_doc_order(result_df)

    hit = int(any(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids))
    recall = sum(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids) / len(relevant_doc_ids)

    first_relevant_rank = None
    for idx, doc_id in enumerate(predicted_doc_ids, start=1):
        if doc_id in relevant_doc_ids:
            first_relevant_rank = idx
            break

    mrr = 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank

    return {
        "predicted_doc_ids": predicted_doc_ids,
        "hit": hit,
        "recall": recall,
        "first_relevant_rank": first_relevant_rank,
        "mrr": mrr,
        "result_df": result_df,
    }


def evaluate_benchmark(
    benchmark_rows: List[Dict[str, object]],
    artifacts: RetrieverArtifacts,
    top_k: int = 3,
) -> pd.DataFrame:
    rows = []
    for row in benchmark_rows:
        metrics = evaluate_query(
            query=row["query"],
            relevant_doc_ids=row["relevant_doc_ids"],
            artifacts=artifacts,
            top_k=top_k,
        )
        rows.append(
            {
                "query_id": row["query_id"],
                "query": row["query"],
                "relevant_doc_ids": ", ".join(row["relevant_doc_ids"]),
                "predicted_doc_ids": ", ".join(metrics["predicted_doc_ids"]),
                f"hit@{top_k}": metrics["hit"],
                f"recall@{top_k}": metrics["recall"],
                f"MRR@{top_k}": metrics["mrr"],
                "first_relevant_rank": metrics["first_relevant_rank"],
            }
        )
    return pd.DataFrame(rows)

# Контрольные запросы и оценка retrieval

In [ ]:
benchmark_queries: List[Dict[str, object]] = [
    {
        "query_id": "q01",
        "query": "What is the purpose of Automake?",
        "relevant_doc_ids": ["doc_0"],
    },
    {
        "query_id": "q02",
        "query": "What are the Autotools?",
        "relevant_doc_ids": ["doc_0","doc_1","doc_3"],
    },
    {
        "query_id": "q03",
        "query": "What does make installcheck do?",
        "relevant_doc_ids": ["doc_2","doc_5"],
    },
    {
        "query_id": "q04",
        "query": "How many trees tdoes the GNU Build System distinguish?",
        "relevant_doc_ids": ["doc_8"],
    },
    {
        "query_id": "q05",
        "query": "What is a VPATH build?",
        "relevant_doc_ids": ["doc_8"],
    },
    {
        "query_id": "q06",
        "query": "How does Automake handle comments in a Makefile.am file?",
        "relevant_doc_ids": ["doc_4"],
    },
    {
        "query_id": "q07",
        "query": "What are the --build, --host, and --target configure options used for?",
        "relevant_doc_ids": ["doc_9"],
    },
    {
        "query_id": "q08",
        "query": "What is the role of the configure script in the GNU Build System?",
        "relevant_doc_ids": ["doc_1","doc_2","doc_8"],
    },
]
benchmark_df = pd.DataFrame(benchmark_queries)
display(benchmark_df)

In [ ]:
# Прогоняем весь benchmark и считаем метрики для baseline-конфигурации.
baseline_eval_k3 = evaluate_benchmark(benchmark_queries, artifacts=artifacts, top_k=3)
display(baseline_eval_k3)

summary_k3 = pd.DataFrame(
    {
        "metric": ["mean_hit@3", "mean_recall@3", "mean_MRR@3"],
        "value": [
            baseline_eval_k3["hit@3"].mean(),
            baseline_eval_k3["recall@3"].mean(),
            baseline_eval_k3["MRR@3"].mean(),
        ],
    }
)
display(summary_k3)
summary_k3_save_blobb = pd.DataFrame(
    {
        "query": benchmark_df["query"],
        "expected_source": baseline_eval_k3["relevant_doc_ids"],
        "retrieved_sources": baseline_eval_k3["predicted_doc_ids"],
        "hit_at_k": baseline_eval_k3["hit@3"],
    }
)
display(summary_k3_save_blobb)
summary_k3_save_blobb.to_csv("artifacts/retrieval_eval.csv", index=False)

# Небольшой эксперимент с параметрами retrieval

In [ ]:
# Фиксируем baseline-конфигурацию и смотрим, как меняются метрики при разных top_k.
topk_rows = []

for top_k in [1, 2, 3, 4, 5]:
    eval_df = evaluate_benchmark(benchmark_queries, artifacts=artifacts, top_k=top_k)
    topk_rows.append(
        {
            "top_k": top_k,
            "mean_hit": eval_df[f"hit@{top_k}"].mean(),
            "mean_recall": eval_df[f"recall@{top_k}"].mean(),
            "mean_MRR": eval_df[f"MRR@{top_k}"].mean(),
        }
    )

topk_df = pd.DataFrame(topk_rows)
display(topk_df)

plt.figure(figsize=(8, 4))
plt.plot(topk_df["top_k"], topk_df["mean_hit"], marker="o", label="mean_hit")
plt.plot(topk_df["top_k"], topk_df["mean_recall"], marker="o", label="mean_recall")
plt.plot(topk_df["top_k"], topk_df["mean_MRR"], marker="o", label="mean_MRR")
plt.xlabel("top_k")
plt.ylabel("значение метрики")
plt.title("Как выбор top_k влияет на retrieval")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Обновление базы знаний и переиндексация

In [ ]:
new_documents: List[Dict[str, str]] = [ {"doc_id":f"doc_{i+10}", "title":text_labels_added[i], "text": texts_added[i]} for i in range(len(text_labels_added))]

In [ ]:
updated_documents = documents + new_documents

display(pd.DataFrame(new_documents)[["doc_id", "title","text"]])
display(pd.DataFrame(updated_documents)[["doc_id", "title"]])
new_queries = [
    "How to enable dependency tracking using in GNU Autotools?",
    "Which version of GNU Automake requires gcc and GNU make to track dependencies automatically?",
]

display(Markdown("### Как baseline-база отвечает на новые запросы"))
for query in new_queries:
    display(Markdown(f"**Запрос:** {query}"))
    f = search_chunks(query, artifacts=artifacts, top_k=3)
    display(f[["rank", "score", "doc_id", "title", "chunk_text"]])


In [ ]:
# Переиндексируем корпус уже с новыми документами.
updated_artifacts = build_retriever(
    updated_documents,
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP,
    device=DEVICE,
)

# Расширяем benchmark новыми запросами.
extended_benchmark_queries = benchmark_queries + [
    {
        "query_id": "q09",
        "query": "How to enable dependency tracking using in GNU Autotools?",
        "relevant_doc_ids": ["doc_10","doc_11"],
    },
    {
        "query_id": "q10",
        "query": "Which version of GNU Automake requires gcc and GNU make to track dependencies automatically?",
        "relevant_doc_ids": ["doc_10","doc_11"],
    },
]

before_update_eval = evaluate_benchmark(extended_benchmark_queries, artifacts=artifacts, top_k=3)
after_update_eval = evaluate_benchmark(extended_benchmark_queries, artifacts=updated_artifacts, top_k=3)

comparison_df = before_update_eval.merge(
    after_update_eval,
    on=["query_id", "query", "relevant_doc_ids"],
    suffixes=("_before", "_after"),
)

display(comparison_df)

summary_comparison_df = pd.DataFrame(
    {
        "metric": ["mean_hit@3", "mean_recall@3", "mean_MRR@3"],
        "before_update": [
            before_update_eval["hit@3"].mean(),
            before_update_eval["recall@3"].mean(),
            before_update_eval["MRR@3"].mean(),
        ],
        "after_update": [
            after_update_eval["hit@3"].mean(),
            after_update_eval["recall@3"].mean(),
            after_update_eval["MRR@3"].mean(),
        ],
    }
)
summary_comparison_df["delta"] = summary_comparison_df["after_update"] - summary_comparison_df["before_update"]
display(summary_comparison_df)

display(Markdown("### Как updated-база отвечает на новые запросы"))
for query in new_queries:
    display(Markdown(f"**Запрос:** {query}"))
    display(search_chunks(query, artifacts=updated_artifacts, top_k=3)[["rank", "score", "doc_id", "title", "chunk_text"]])

In [ ]:
print(comparison_df.columns)
update_df_save_blobb = pd.DataFrame(
    {
        "query":comparison_df["query"],
        "before_retrieved_sources":comparison_df["predicted_doc_ids_before"],
        "after_retrieved_sources":comparison_df["predicted_doc_ids_after"],
        "changed":comparison_df["predicted_doc_ids_before"] != comparison_df["predicted_doc_ids_after"],
    }
)
update_df_save_blobb.to_csv("artifacts/retrieval_before_after_update.csv", index=False)
display(update_df_save_blobb)

# Mini-RAG

In [ ]:
def split_into_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]


def pick_best_sentences(query: str, text: str, top_n: int = 2) -> List[str]:
    sentences = split_into_sentences(text)
    if not sentences:
        return []

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([query] + sentences).toarray().astype(np.float32)

    query_vec = matrix[0]
    sentence_vecs = matrix[1:]

    query_norm = np.linalg.norm(query_vec) + 1e-12
    sent_norms = np.linalg.norm(sentence_vecs, axis=1) + 1e-12
    scores = (sentence_vecs @ query_vec) / (sent_norms * query_norm)

    best_idx = np.argsort(-scores)[:top_n]
    return [sentences[i] for i in best_idx if scores[i] > 0]


def answer_without_retrieval(query: str, documents: List[Dict[str, str]]) -> Dict[str, object]:
    doc_texts = [doc["title"] + ". " + doc["text"] for doc in documents]
    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform(doc_texts + [query]).toarray().astype(np.float32)

    doc_vecs = matrix[:-1]
    query_vec = matrix[-1]

    doc_norms = np.linalg.norm(doc_vecs, axis=1) + 1e-12
    query_norm = np.linalg.norm(query_vec) + 1e-12
    scores = (doc_vecs @ query_vec) / (doc_norms * query_norm)

    best_idx = int(np.argmax(scores))
    best_doc = documents[best_idx]
    best_sentences = pick_best_sentences(query, best_doc["text"], top_n=2)

    if best_sentences:
        answer = " ".join(best_sentences)
    else:
        answer = (
            "Не удалось уверенно извлечь ответ без retrieval по чанкам. "
            "Система выбрала наиболее похожий документ целиком."
        )

    return {
        "answer": answer,
        "selected_doc_id": best_doc["doc_id"],
        "selected_title": best_doc["title"],
        "score": float(scores[best_idx]),
    }

In [ ]:
baseline_example = answer_without_retrieval(
    "What are the --build and --host configure options used for?",
    documents,
)

display(pd.DataFrame([baseline_example]))

In [ ]:

def build_context_from_retrieval(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> Tuple[str, pd.DataFrame]:
    retrieved = search_chunks(query, artifacts=artifacts, top_k=top_k)
    context_blocks = []

    for _, row in retrieved.iterrows():
        block = (
            f"[Источник: {row['doc_id']} | {row['title']} | score={row['score']:.4f}]\n"
            f"{row['chunk_text']}"
        )
        context_blocks.append(block)

    context = "\n\n".join(context_blocks)
    return context, retrieved

In [ ]:
query = "What are the --build and --host configure options used for?"
context, retrieved_df = build_context_from_retrieval(query, artifacts=artifacts, top_k=3)

display(Markdown(f"### Запрос: {query}"))
display(retrieved_df)
print(context)

In [ ]:
def generate_answer_from_context(query: str, context: str, max_sentences: int = 2) -> str:
    # Убираем технические строки источников из ранжирования, но не из общего контекста.
    raw_lines = [line.strip() for line in context.splitlines() if line.strip()]
    content_lines = [line for line in raw_lines if not line.startswith("[Источник:")]

    sentence_pool = []
    for line in content_lines:
        sentence_pool.extend(split_into_sentences(line))

    sentence_pool = [s for s in sentence_pool if len(s.split()) >= 4]

    if not sentence_pool:
        return "Недостаточно контекста для построения ответа."

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([query] + sentence_pool).toarray().astype(np.float32)

    query_vec = matrix[0]
    sentence_vecs = matrix[1:]

    query_norm = np.linalg.norm(query_vec) + 1e-12
    sent_norms = np.linalg.norm(sentence_vecs, axis=1) + 1e-12
    scores = (sentence_vecs @ query_vec) / (sent_norms * query_norm)

    ranked_idx = np.argsort(-scores)
    selected_sentences = []
    used_normalized = set()

    for idx in ranked_idx:
        sentence = sentence_pool[idx]
        normalized = sentence.lower().strip()
        if scores[idx] <= 0:
            continue
        if normalized in used_normalized:
            continue
        used_normalized.add(normalized)
        selected_sentences.append(sentence)
        if len(selected_sentences) >= max_sentences:
            break

    if not selected_sentences:
        return "В найденном контексте нет достаточно релевантного фрагмента для уверенного ответа."

    return " ".join(selected_sentences)

In [ ]:
answer_example = generate_answer_from_context(query, context)
print(answer_example)

In [ ]:
def mini_rag_answer(
    query: str,
    artifacts: RetrieverArtifacts,
    top_k: int = 3,
    max_answer_sentences: int = 2,
) -> Dict[str, object]:
    context, retrieved = build_context_from_retrieval(query, artifacts=artifacts, top_k=top_k)
    answer = generate_answer_from_context(query, context=context, max_sentences=max_answer_sentences)

    return {
        "query": query,
        "answer": answer,
        "context": context,
        "sources": retrieved,
    }

In [ ]:
rag_result = mini_rag_answer(
    "What is the three-way cross-compilation called ",
    artifacts=artifacts,
    top_k=3,
)

display(Markdown(f"### Вопрос: {rag_result['query']}"))
display(Markdown(f"**Ответ:** {rag_result['answer']}"))
display(Markdown("**Источники:**"))
display(rag_result["sources"])

In [ ]:
comparison_queries = [
    "What is the purpose of Automake?",
    "What does make installcheck do?",
    "What are the --build, --host, and --target configure options used for?",
    "What is the role of the configure script in the GNU Build System?",
    "What is a VPATH build?",
]

comparison_rows = []

for query in comparison_queries:
    baseline = answer_without_retrieval(query, documents)
    rag = mini_rag_answer(query, artifacts=artifacts, top_k=3)

    comparison_rows.append(
        {
            "query": query,
            "baseline_doc_id": baseline["selected_doc_id"],
            "baseline_score": baseline["score"],
            "baseline_answer": baseline["answer"],
            "rag_answer": rag["answer"],
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

In [ ]:
for _, row in comparison_df.iterrows():
    display(Markdown(f"## Вопрос: {row['query']}"))
    display(Markdown(f"**Baseline без retrieval:** {row['baseline_answer']}"))
    display(Markdown(f"**Mini-RAG:** {row['rag_answer']}"))

In [ ]:
query = "What is a VPATH build?"
rag_result = mini_rag_answer(query, artifacts=artifacts, top_k=3)

display(Markdown(f"### Вопрос: {query}"))
display(Markdown(f"**Ответ:** {rag_result['answer']}"))
display(rag_result["sources"][["rank", "score", "doc_id", "title", "chunk_text"]])

In [ ]:
qa_benchmark = [
    {
        "query_id": "q01",
        "query": "What is the purpose of Automake?",
        "relevant_doc_ids": ["doc_0"],
        "expected_keywords": ["Automake", "tool", "generating"],
    },
    {
        "query_id": "q02",
        "query": "What are the Autotools?",
        "relevant_doc_ids": ["doc_0","doc_1","doc_3"],
        "expected_keywords": ["Automake", "configure", "System"],
    },
    {
        "query_id": "q03",
        "query": "What does make installcheck do?",
        "relevant_doc_ids": ["doc_2","doc_5"],
        "expected_keywords": ["tests", "nothing"],
    },
    {
        "query_id": "q04",
        "query": "What is the role of the configure script in the GNU Build System?",
        "relevant_doc_ids": ["doc_1","doc_2","doc_8"],
        "expected_keywords": ["configure", "probes", "features", "Makefile"],
    },
    {
        "query_id": "q05",
        "query": "What are the --build, --host, and --target configure options used for?",
        "relevant_doc_ids": ["doc_9"],
        "expected_keywords": ["host", "target", "build"],
    },
]

def keyword_recall(answer: str, expected_keywords: List[str]) -> float:
    answer_lower = answer.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in answer_lower)
    return hits / len(expected_keywords) if expected_keywords else np.nan


def evaluate_mini_rag(
    benchmark_rows: List[Dict[str, object]],
    artifacts: RetrieverArtifacts,
    top_k: int = 3,
) -> pd.DataFrame:
    rows = []

    for item in benchmark_rows:
        query = item["query"]
        relevant_doc_ids = item["relevant_doc_ids"]
        expected_keywords = item["expected_keywords"]

        retrieved = search_chunks(query, artifacts=artifacts, top_k=top_k)
        predicted_doc_ids = retrieved["doc_id"].tolist()
        retrieval_hit = int(any(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids))

        baseline = answer_without_retrieval(query, documents)
        rag = mini_rag_answer(query, artifacts=artifacts, top_k=top_k)

        rows.append(
            {
                "query_id": item["query_id"],
                "query": query,
                "relevant_doc_ids": ", ".join(relevant_doc_ids),
                "predicted_doc_ids": ", ".join(predicted_doc_ids),
                f"retrieval_hit@{top_k}": retrieval_hit,
                "baseline_keyword_recall": keyword_recall(baseline["answer"], expected_keywords),
                "rag_keyword_recall": keyword_recall(rag["answer"], expected_keywords),
                "baseline_answer": baseline["answer"],
                "rag_answer": rag["answer"],
            }
        )

    return pd.DataFrame(rows)

In [ ]:
evaluation_df = evaluate_mini_rag(qa_benchmark, artifacts=artifacts, top_k=3)
display(evaluation_df)

In [ ]:
summary_df = pd.DataFrame(
    {
        "metric": ["mean_retrieval_hit@3", "mean_baseline_keyword_recall", "mean_rag_keyword_recall"],
        "value": [
            evaluation_df["retrieval_hit@3"].mean(),
            evaluation_df["baseline_keyword_recall"].mean(),
            evaluation_df["rag_keyword_recall"].mean(),
        ],
    }
)
display(summary_df)
rag_df_save_blobb = pd.DataFrame(
    {
        "question":evaluation_df["query"],
        "answer":evaluation_df["rag_answer"],
        "retrieved_sources":evaluation_df["predicted_doc_ids"],
    }
)
rag_df_save_blobb.to_csv("artifacts/rag_examples.csv", index=False)
display(rag_df_save_blobb)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(
    evaluation_df["query_id"],
    evaluation_df["baseline_keyword_recall"],
    marker="o",
    label="baseline_keyword_recall",
)
plt.plot(
    evaluation_df["query_id"],
    evaluation_df["rag_keyword_recall"],
    marker="o",
    label="rag_keyword_recall",
)
plt.ylim(0, 1.05)
plt.xlabel("query_id")
plt.ylabel("значение метрики")
plt.title("Baseline vs mini-RAG по keyword recall")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# Анализ плохих кейсов

In [ ]:
weak_cases = evaluation_df[
    evaluation_df["rag_keyword_recall"] < evaluation_df["rag_keyword_recall"].max()
].reset_index(drop=True)

display(weak_cases if len(weak_cases) > 0 else evaluation_df.head(0))

if len(weak_cases) == 0:
    print("На этом маленьком benchmark все ответы mini-RAG достигли одинакового keyword recall.")
else:
    print("Эти вопросы стоит разобрать вручную: проблема может быть в retrieval, чанкинге или самом генераторе ответа.")

In [ ]:
for _, row in weak_cases.iterrows():
    display(Markdown(f"### Слабый кейс: {row['query']}"))
    rag = mini_rag_answer(row["query"], artifacts=artifacts, top_k=3)
    display(Markdown(f"**Ответ mini-RAG:** {rag['answer']}"))
    display(rag["sources"][["rank", "score", "doc_id", "title", "chunk_text"]])

### Из 4 кейсов - второй и четвертый нормальные, но в четвертом не совсем полный ответ на вопрос